In [16]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic

In [17]:
client=Anthropic()
model = "claude-haiku-4-5"

In [18]:
def add_user_message(messages, text):
    user_message={'role':'user', 'content':text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={'role':'assistant', 'content':text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=[]):
    params={
        'model':model,
        'max_tokens':1000,
        'messages':messages
    }
    if system is not None:
        params['system']=system
    if temperature is not None:
        params['temperature']=temperature
    if stop_sequences:
        params['stop_sequences']=stop_sequences

    message=client.messages.create(**params)
    return message.content[0].text

In [19]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text=chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset=generate_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., 's3://my-bucket-us-east-1/path/to/file'). The function should return the region code."},
 {'task': 'Create a JSON CloudFormation template snippet that defines an IAM role with a trust relationship allowing the EC2 service to assume it.'},
 {'task': "Write a regular expression that validates an AWS ARN format and captures the service, resource type, and resource ID components (e.g., 'arn:aws:s3:::my-bucket')."}]

In [22]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [23]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [24]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [25]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [26]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [27]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_uri(s3_uri: str) -> Optional[str]:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/path' or 's3://bucket-name-region/path'\n    \n    Returns:\n        AWS region code (e.g., 'us-east-1') or None if not found\n    \n    Examples:\n        >>> extract_region_from_s3_uri('s3://my-bucket-us-east-1/path/to/file')\n        'us-east-1'\n        >>> extract_region_from_s3_uri('s3://data-eu-west-1/file.txt')\n        'eu-west-1'\n    \"\"\"\n    # List of all valid AWS regions\n    valid_regions = {\n        'us-east-1', 'us-east-2', 'us-west-1', 'us-west-2',\n        'eu-west-1', 'eu-west-2', 'eu-west-3', 'eu-central-1', 'eu-north-1',\n        'ap-southeast-1', 'ap-southeast-2', 'ap-northeast-1', 'ap-northeast-2